In [1]:
import csv
from datetime import datetime
from pathlib import Path
import pandas as pd

# Paths relative to data_labeling/eda/
BASE_DIR = Path("..").resolve()

# Verified source files
GOES_CSV = (
    BASE_DIR
    / "data_labels/GOES/NEW_full_dataset_cleaned_1_hours_with_loc_and_time_new.csv"
)
NOAA_CSV = (
    BASE_DIR / "data_labels/noaa_2010_2026/labels_noaa_2010_2026.csv"
)  # Updated folder name

# Output destination matching your new folder
OUTPUT_DIR = (
    BASE_DIR / "data_labels/GOES_2010_2018_NOAA_2019_2026/CL_Labels"
)  # Target folder
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("GOES source exists:", GOES_CSV.exists())
print("NOAA source exists:", NOAA_CSV.exists())
print("Output directory ready at:", OUTPUT_DIR.resolve())

GOES source exists: True
NOAA source exists: True
Output directory ready at: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/GOES_2010_2018_NOAA_2019_2026/CL_Labels


In [2]:
def to_binary_flare(goes_class):
  gc = str(goes_class).strip().upper()
  if gc.startswith("X"):
    return 1
  if gc.startswith("M"):
    try:
      return 1 if float(gc[1:]) >= 1.0 else 0
    except ValueError:
      return 1
  return 0

In [4]:
assert to_binary_flare("M1.0") == 1
assert to_binary_flare("M0.9") == 0
assert to_binary_flare("X1.5") == 1
assert to_binary_flare("C4.2") == 0
assert to_binary_flare("NF") == 0
# assert to_binary_flare("M0.9") == 1
print("Binary conversion function verified!")

Binary conversion function verified!


Load NOAA Source

In [5]:
df_noaa = pd.read_csv(NOAA_CSV)
print("Raw NOAA rows loaded:", len(df_noaa))

Raw NOAA rows loaded: 128117


In [6]:
df_noaa

,label,goes_class,fl_lon,fl_lat,rest_fl,rest_lon,rest_lat
0,2010/12/06/HMI.m2010.12.06_07.00.00.jpg,NF,unk,unk,NaN,NaN,NaN
1,2010/12/06/HMI.m2010.12.06_08.00.00.jpg,B1.9,UNK,UNK,NaN,NaN,NaN
2,2010/12/06/HMI.m2010.12.06_09.00.00.jpg,B1.9,UNK,UNK,NaN,NaN,NaN
3,2010/12/06/HMI.m2010.12.06_10.00.00.jpg,B1.9,UNK,UNK,NaN,NaN,NaN
4,2010/12/06/HMI.m2010.12.06_11.00.00.jpg,B1.9,UNK,UNK,NaN,NaN,NaN
...,...,...,...,...,...,...,...
128112,2026/07/31/HMI.m2026.07.31_19.00.00.jpg,NF,unk,unk,NaN,NaN,NaN
128113,2026/07/31/HMI.m2026.07.31_20.00.00.jpg,NF,unk,unk,NaN,NaN,NaN
128114,2026/07/31/HMI.m2026.07.31_21.00.00.jpg,NF,unk,unk,NaN,NaN,NaN
128115,2026/07/31/HMI.m2026.07.31_22.00.00.jpg,NF,unk,unk,NaN,NaN,NaN


In [7]:
# Parse datetimes
df_noaa["datetime_str"] = df_noaa["label"].apply(
    lambda x: x.split("/")[-1].replace("HMI.m", "").replace(".jpg", "")
)

df_noaa

,label,goes_class,fl_lon,fl_lat,rest_fl,rest_lon,rest_lat,datetime_str
0,2010/12/06/HMI.m2010.12.06_07.00.00.jpg,NF,unk,unk,NaN,NaN,NaN,2010.12.06_07.00.00
1,2010/12/06/HMI.m2010.12.06_08.00.00.jpg,B1.9,UNK,UNK,NaN,NaN,NaN,2010.12.06_08.00.00
2,2010/12/06/HMI.m2010.12.06_09.00.00.jpg,B1.9,UNK,UNK,NaN,NaN,NaN,2010.12.06_09.00.00
3,2010/12/06/HMI.m2010.12.06_10.00.00.jpg,B1.9,UNK,UNK,NaN,NaN,NaN,2010.12.06_10.00.00
4,2010/12/06/HMI.m2010.12.06_11.00.00.jpg,B1.9,UNK,UNK,NaN,NaN,NaN,2010.12.06_11.00.00
...,...,...,...,...,...,...,...,...
128112,2026/07/31/HMI.m2026.07.31_19.00.00.jpg,NF,unk,unk,NaN,NaN,NaN,2026.07.31_19.00.00
128113,2026/07/31/HMI.m2026.07.31_20.00.00.jpg,NF,unk,unk,NaN,NaN,NaN,2026.07.31_20.00.00
128114,2026/07/31/HMI.m2026.07.31_21.00.00.jpg,NF,unk,unk,NaN,NaN,NaN,2026.07.31_21.00.00
128115,2026/07/31/HMI.m2026.07.31_22.00.00.jpg,NF,unk,unk,NaN,NaN,NaN,2026.07.31_22.00.00


In [8]:
df_noaa["datetime"] = pd.to_datetime(
    df_noaa["datetime_str"], format="%Y.%m.%d_%H.%M.%S"
)
df_noaa["year"] = df_noaa["datetime"].dt.year

df_noaa

,label,goes_class,fl_lon,fl_lat,rest_fl,rest_lon,rest_lat,datetime_str,datetime,year
0,2010/12/06/HMI.m2010.12.06_07.00.00.jpg,NF,unk,unk,NaN,NaN,NaN,2010.12.06_07.00.00,2010-12-06 07:00:00,2010
1,2010/12/06/HMI.m2010.12.06_08.00.00.jpg,B1.9,UNK,UNK,NaN,NaN,NaN,2010.12.06_08.00.00,2010-12-06 08:00:00,2010
2,2010/12/06/HMI.m2010.12.06_09.00.00.jpg,B1.9,UNK,UNK,NaN,NaN,NaN,2010.12.06_09.00.00,2010-12-06 09:00:00,2010
3,2010/12/06/HMI.m2010.12.06_10.00.00.jpg,B1.9,UNK,UNK,NaN,NaN,NaN,2010.12.06_10.00.00,2010-12-06 10:00:00,2010
4,2010/12/06/HMI.m2010.12.06_11.00.00.jpg,B1.9,UNK,UNK,NaN,NaN,NaN,2010.12.06_11.00.00,2010-12-06 11:00:00,2010
...,...,...,...,...,...,...,...,...,...,...
128112,2026/07/31/HMI.m2026.07.31_19.00.00.jpg,NF,unk,unk,NaN,NaN,NaN,2026.07.31_19.00.00,2026-07-31 19:00:00,2026
128113,2026/07/31/HMI.m2026.07.31_20.00.00.jpg,NF,unk,unk,NaN,NaN,NaN,2026.07.31_20.00.00,2026-07-31 20:00:00,2026
128114,2026/07/31/HMI.m2026.07.31_21.00.00.jpg,NF,unk,unk,NaN,NaN,NaN,2026.07.31_21.00.00,2026-07-31 21:00:00,2026
128115,2026/07/31/HMI.m2026.07.31_22.00.00.jpg,NF,unk,unk,NaN,NaN,NaN,2026.07.31_22.00.00,2026-07-31 22:00:00,2026


In [9]:
# Create set of all actually downloaded image paths
downloaded_paths = set(df_noaa["label"])
print(f"Total downloaded and existing images: {len(downloaded_paths):,}")

Total downloaded and existing images: 128,117


In [23]:
# Filter NOAA starting seamlessly from Dec 31, 2018 onwards
df_noaa_seamless = df_noaa[df_noaa["datetime"] >= "2018-12-31 00:00:00"].copy()
df_noaa_seamless["binary"] = df_noaa_seamless["goes_class"].apply(
    to_binary_flare
)

In [24]:
print(
    f"\nNOAA seamless samples (Dec 31, 2018 - 2026):"
    f" {len(df_noaa_seamless):,} (from {df_noaa_seamless['datetime'].min()} to"
    f" {df_noaa_seamless['datetime'].max()})"
)
print("NOAA class counts:")
print(df_noaa_seamless["binary"].value_counts())


NOAA seamless samples (Dec 31, 2018 - 2026): 64,832 (from 2018-12-31 00:00:00 to 2026-07-31 23:00:00)
NOAA class counts:
binary
0    46808
1    18024
Name: count, dtype: int64


Load GOES Source (2010–2018)

In [14]:
df_goes = pd.read_csv(GOES_CSV)
print("Raw GOES rows loaded:", len(df_goes))

Raw GOES rows loaded: 63649


In [15]:
df_goes

,label,goes_class,fl_lon,fl_lat,rest_fl,rest_lon,rest_lat
0,2010/12/06/HMI.m2010.12.06_07.00.00.jpg,NF,unk,unk,NaN,NaN,NaN
1,2010/12/06/HMI.m2010.12.06_08.00.00.jpg,B1.9,-38.5479308163,16.0,['B1.9'],[-38.5479308163],[16.0]
2,2010/12/06/HMI.m2010.12.06_09.00.00.jpg,B1.9,-38.5479308163,16.0,['B1.9'],[-38.5479308163],[16.0]
3,2010/12/06/HMI.m2010.12.06_10.00.00.jpg,B1.9,-38.5479308163,16.0,['B1.9'],[-38.5479308163],[16.0]
4,2010/12/06/HMI.m2010.12.06_11.00.00.jpg,B1.9,-38.5479308163,16.0,['B1.9'],[-38.5479308163],[16.0]
...,...,...,...,...,...,...,...
63644,2018/12/30/HMI.m2018.12.30_19.00.00.jpg,NF,unk,unk,NaN,NaN,NaN
63645,2018/12/30/HMI.m2018.12.30_20.00.00.jpg,NF,unk,unk,NaN,NaN,NaN
63646,2018/12/30/HMI.m2018.12.30_21.00.00.jpg,NF,unk,unk,NaN,NaN,NaN
63647,2018/12/30/HMI.m2018.12.30_22.00.00.jpg,NF,unk,unk,NaN,NaN,NaN


In [16]:
df_goes["datetime_str"] = df_goes["label"].apply(
    lambda x: x.split("/")[-1].replace("HMI.m", "").replace(".jpg", "")
)
df_goes["datetime"] = pd.to_datetime(
    df_goes["datetime_str"], format="%Y.%m.%d_%H.%M.%S"
)
df_goes["year"] = df_goes["datetime"].dt.year

df_goes

,label,goes_class,fl_lon,fl_lat,rest_fl,rest_lon,rest_lat,datetime_str,datetime,year
0,2010/12/06/HMI.m2010.12.06_07.00.00.jpg,NF,unk,unk,NaN,NaN,NaN,2010.12.06_07.00.00,2010-12-06 07:00:00,2010
1,2010/12/06/HMI.m2010.12.06_08.00.00.jpg,B1.9,-38.5479308163,16.0,['B1.9'],[-38.5479308163],[16.0],2010.12.06_08.00.00,2010-12-06 08:00:00,2010
2,2010/12/06/HMI.m2010.12.06_09.00.00.jpg,B1.9,-38.5479308163,16.0,['B1.9'],[-38.5479308163],[16.0],2010.12.06_09.00.00,2010-12-06 09:00:00,2010
3,2010/12/06/HMI.m2010.12.06_10.00.00.jpg,B1.9,-38.5479308163,16.0,['B1.9'],[-38.5479308163],[16.0],2010.12.06_10.00.00,2010-12-06 10:00:00,2010
4,2010/12/06/HMI.m2010.12.06_11.00.00.jpg,B1.9,-38.5479308163,16.0,['B1.9'],[-38.5479308163],[16.0],2010.12.06_11.00.00,2010-12-06 11:00:00,2010
...,...,...,...,...,...,...,...,...,...,...
63644,2018/12/30/HMI.m2018.12.30_19.00.00.jpg,NF,unk,unk,NaN,NaN,NaN,2018.12.30_19.00.00,2018-12-30 19:00:00,2018
63645,2018/12/30/HMI.m2018.12.30_20.00.00.jpg,NF,unk,unk,NaN,NaN,NaN,2018.12.30_20.00.00,2018-12-30 20:00:00,2018
63646,2018/12/30/HMI.m2018.12.30_21.00.00.jpg,NF,unk,unk,NaN,NaN,NaN,2018.12.30_21.00.00,2018-12-30 21:00:00,2018
63647,2018/12/30/HMI.m2018.12.30_22.00.00.jpg,NF,unk,unk,NaN,NaN,NaN,2018.12.30_22.00.00,2018-12-30 22:00:00,2018


In [17]:
downloaded_paths = set(df_noaa["label"])

In [25]:
df_goes_filtered = df_goes[
    (df_goes["datetime"] <= "2018-12-30 23:00:00")
    & (df_goes["label"].isin(downloaded_paths))
].copy()

In [26]:
df_goes_filtered["binary"] = df_goes_filtered["goes_class"].apply(
    to_binary_flare
)

In [27]:
print(
    f"\nGOES valid samples (2010 - Dec 30, 2018): {len(df_goes_filtered):,}"
    f" (from {df_goes_filtered['datetime'].min()} to"
    f" {df_goes_filtered['datetime'].max()})"
)
print("GOES class counts:")
print(df_goes_filtered["binary"].value_counts())


GOES valid samples (2010 - Dec 30, 2018): 63,284 (from 2010-12-06 07:00:00 to 2018-12-30 23:00:00)
GOES class counts:
binary
0    54309
1     8975
Name: count, dtype: int64


Combine into ONE Clean Complete Binary Label File & Save

In [22]:
# 1. Define all columns to include
cols = [
    "label",
    "goes_class",
    "binary",
    "fl_lon",
    "fl_lat",
    "rest_fl",
    "rest_lon",
    "rest_lat",
    "datetime",
]

In [28]:
# 2. Concatenate seamlessly
master_hybrid = pd.concat(
    [df_goes_filtered[cols], df_noaa_seamless[cols]], ignore_index=True
)

In [29]:
master_hybrid = master_hybrid.sort_values("datetime").reset_index(drop=True)

In [30]:
print(f"Total Unified Continuous Dataset: {len(master_hybrid):,} samples")
print(
    f"Date range: {master_hybrid['datetime'].min()}  --> "
    f" {master_hybrid['datetime'].max()}"
)
print("\nFinal Overall Class Distribution:")
print(master_hybrid["binary"].value_counts())
pct_fl = master_hybrid["binary"].sum() / len(master_hybrid) * 100
print(f"Percentage of Flares (>= M1.0): {pct_fl:.2f}%")

Total Unified Continuous Dataset: 128,116 samples
Date range: 2010-12-06 07:00:00  -->  2026-07-31 23:00:00

Final Overall Class Distribution:
binary
0    101117
1     26999
Name: count, dtype: int64
Percentage of Flares (>= M1.0): 21.07%


In [31]:
# 3. Setup output folder
TARGET_DIR = BASE_DIR / "data_labels/GOES_2010_2018_NOAA_2019_2026"
TARGET_DIR.mkdir(parents=True, exist_ok=True)

In [32]:
# -------------------------------------------------------------
# File A: Complete Metadata Master (with coordinates & all flare lists)
# -------------------------------------------------------------
meta_cols = [
    "label",
    "goes_class",
    "binary_target",
    "fl_lon",
    "fl_lat",
    "rest_fl",
    "rest_lon",
    "rest_lat",
]
complete_meta_df = master_hybrid.rename(columns={"binary": "binary_target"})[
    meta_cols
]
out_meta_file = (
    TARGET_DIR / "labels_goes_2010_2018_noaa_2019_2026_complete_with_loc.csv"
)
complete_meta_df.to_csv(out_meta_file, index=False)
print(
    f"\n1. Saved complete rich metadata master ({len(complete_meta_df):,} rows)"
    f" to:\n   {out_meta_file.name}"
)


1. Saved complete rich metadata master (128,116 rows) to:
   labels_goes_2010_2018_noaa_2019_2026_complete_with_loc.csv


In [33]:
# -------------------------------------------------------------
# File B: Clean Binary File for Model Training (label, goes_class with 0 or 1)
# -------------------------------------------------------------
clean_binary_df = pd.DataFrame(
    {"label": master_hybrid["label"], "goes_class": master_hybrid["binary"]}
)
out_binary_file = (
    TARGET_DIR / "labels_goes_2010_2018_noaa_2019_2026_complete_binary.csv"
)
clean_binary_df.to_csv(out_binary_file, index=False)
print(
    f"\n2. Saved clean binary training file ({len(clean_binary_df):,} rows)"
    f" to:\n   {out_binary_file.name}"
)


2. Saved clean binary training file (128,116 rows) to:
   labels_goes_2010_2018_noaa_2019_2026_complete_binary.csv


In [36]:
import csv
from datetime import datetime, timedelta
import pandas as pd
# Load existing datetimes
df_hybrid = pd.read_csv(
    "../data_labels/GOES_2010_2018_NOAA_2019_2026/labels_goes_2010_2018_noaa_2019_2026_complete_binary.csv"
)
existing_dts = set(
    pd.to_datetime(
        df_hybrid["label"].apply(
            lambda x: x.split("/")[-1].replace("HMI.m", "").replace(".jpg", "")
        ),
        format="%Y.%m.%d_%H.%M.%S",
    )
)
# Build full grid
start_dt = min(existing_dts)
end_dt = max(existing_dts)
full_grid = pd.date_range(start=start_dt, end=end_dt, freq="1h")
# Find missing
missing_dts = sorted(set(full_grid) - existing_dts)
missing_df = pd.DataFrame(
    {
        "missing_timestamp_utc": missing_dts,
        "year": [dt.year for dt in missing_dts],
        "month": [dt.month for dt in missing_dts],
        "day": [dt.day for dt in missing_dts],
        "hour": [dt.hour for dt in missing_dts],
    }
)


missing_df

,missing_timestamp_utc,year,month,day,hour
0,2010-12-07 19:00:00,2010,12,7,19
1,2010-12-08 16:00:00,2010,12,8,16
2,2010-12-08 17:00:00,2010,12,8,17
3,2010-12-08 18:00:00,2010,12,8,18
4,2010-12-08 19:00:00,2010,12,8,19
...,...,...,...,...,...
9080,2026-07-27 07:00:00,2026,7,27,7
9081,2026-07-28 07:00:00,2026,7,28,7
9082,2026-07-29 07:00:00,2026,7,29,7
9083,2026-07-30 07:00:00,2026,7,30,7
